# MultiMed — Train batch

Notebook chạy pipeline trên 500 mẫu của split **train** để kiểm tra ASR/NER.

Kết quả được lưu riêng trong `/content/multimed_results_500_train_rerun/`.
Chọn **Runtime → Change runtime type → T4 GPU** rồi chạy các cell theo thứ tự.


In [1]:
!pip -q install -U accelerate safetensors librosa soundfile jiwer seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [2]:
import csv
import json
import time
from collections import Counter
from pathlib import Path

import torch
from datasets import Audio, load_dataset
from jiwer import wer
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoModelForTokenClassification,
    AutoProcessor,
    AutoTokenizer,
    pipeline,
)

assert torch.cuda.is_available(), (
    'Chưa thấy GPU. Chọn Runtime → Change runtime type → T4 GPU.'
)
DEVICE = 'cuda'
DEVICE_INDEX = 0
TORCH_DTYPE = torch.float16
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


In [3]:
# Cấu hình train batch
ASR_REPO = 'leduckhai/MultiMed-ST'
ASR_SUBFOLDER = 'asr/whisper-small-vietnamese/checkpoint-5000'
ASR_PROCESSOR_SUBFOLDER = 'asr/whisper-small-vietnamese'
NER_REPO = 'leduckhai/VietMed-NER'
NER_SUBFOLDER = 'xlm-roberta-base-VietMed-NER'
DATASET_ID = 'leduckhai/VietMed'
SPLIT = 'train'
START_INDEX = 0
NUM_SAMPLES = 500
MAX_DURATION_SECONDS = 30
OUTPUT_DIR = Path('/content/multimed_results_500_train_rerun')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / 'results.jsonl'
ERROR_PATH = OUTPUT_DIR / 'errors.jsonl'
print(f'Chạy {NUM_SAMPLES} mẫu từ {DATASET_ID}/{SPLIT}, bắt đầu tại index {START_INDEX}')
print(f'Kết quả tạm: {CHECKPOINT_PATH}')

Chạy 500 mẫu từ leduckhai/VietMed/train, bắt đầu tại index 0
Kết quả tạm: /content/multimed_results_500_train_rerun/results.jsonl


In [4]:
# Load model một lần cho toàn batch
asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    ASR_REPO, subfolder=ASR_SUBFOLDER, dtype=TORCH_DTYPE,
    low_cpu_mem_usage=True, use_safetensors=True,
).to(DEVICE)
asr_processor = AutoProcessor.from_pretrained(ASR_REPO, subfolder=ASR_PROCESSOR_SUBFOLDER)
asr_pipe = pipeline(
    'automatic-speech-recognition', model=asr_model,
    tokenizer=asr_processor.tokenizer, feature_extractor=asr_processor.feature_extractor,
    torch_dtype=TORCH_DTYPE, device=DEVICE_INDEX,
)

ner_model = AutoModelForTokenClassification.from_pretrained(
    NER_REPO, subfolder=NER_SUBFOLDER,
).to(DEVICE)
ner_tokenizer = AutoTokenizer.from_pretrained(NER_REPO, subfolder=NER_SUBFOLDER)
ner_pipe = pipeline(
    'ner', model=ner_model, tokenizer=ner_tokenizer,
    aggregation_strategy='simple', device=DEVICE_INDEX,
)
print('ASR và NER models loaded')

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

asr/whisper-small-vietnamese/checkpoint-(…): reconstructing file:   0%|          |  0.00B /  967MB            

asr/whisper-small-vietnamese/checkpoint-(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.00k [00:00<?, ?B/s]

...

xlm-roberta-base-VietMed-NER/sentencepie(…): downloading bytes:           |  0.00B            

xlm-roberta-base-VietMed-NER/tokenizer.j(…): reconstructing file:   0%|          |  0.00B / 17.1MB            

xlm-roberta-base-VietMed-NER/tokenizer.j(…): downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

ASR và NER models loaded


In [5]:
# Lấy các mẫu audio cần chạy; streaming tránh tải toàn bộ dataset
# Vòng lặp lấy đủ NUM_SAMPLES mẫu hợp lệ, không chỉ giới hạn trong START_INDEX + NUM_SAMPLES.
dataset = load_dataset(DATASET_ID, split=SPLIT, streaming=True)
dataset = dataset.cast_column('audio', Audio(sampling_rate=16000, decode=True))
rows = []
target_start = START_INDEX
index = -1
for index, row in enumerate(dataset):
    if index < target_start:
        continue
    duration = row.get('duration')
    if duration is not None and MAX_DURATION_SECONDS and duration > MAX_DURATION_SECONDS:
        continue
    rows.append((index, row))
    if len(rows) >= NUM_SAMPLES:
        break
print('Số mẫu thực tế:', len(rows))
print('Index cuối đã nạp:', index)

README.md:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

Số mẫu thực tế: 500
Index cuối đã nạp: 499


In [6]:
def extract_audio(row):
    import io
    import soundfile as sf

    audio = row['audio']
    if hasattr(audio, 'get_all_samples'):
        decoded = audio.get_all_samples()
        return decoded.data.squeeze().cpu().numpy(), int(decoded.sample_rate)
    if isinstance(audio, dict) and 'array' in audio and 'sampling_rate' in audio:
        return audio['array'], int(audio['sampling_rate'])
    if isinstance(audio, dict) and audio.get('bytes'):
        audio_array, audio_rate = sf.read(io.BytesIO(audio['bytes']), dtype='float32')
        if getattr(audio_array, 'ndim', 1) > 1:
            audio_array = audio_array.mean(axis=1)
        return audio_array, int(audio_rate)
    raise ValueError('Không giải mã được audio của mẫu dataset')

def extract_entities(text):
    entities = []
    for item in ner_pipe(text):
        label = item.get('entity_group') or item.get('entity')
        if label in {'0', 'O', 'dum'}:
            continue
        entities.append({
            'text': item['word'], 'label': label,
            'score': round(float(item['score']), 6),
            'start': int(item['start']), 'end': int(item['end']),
        })
    return entities

results_by_index = {}
if CHECKPOINT_PATH.exists():
    with CHECKPOINT_PATH.open(encoding='utf-8') as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                results_by_index[item['index']] = item
print('Đã có kết quả checkpoint:', len(results_by_index))

pending_rows = [(index, row) for index, row in rows if index not in results_by_index]
started = time.time()
with CHECKPOINT_PATH.open('a', encoding='utf-8') as result_file, ERROR_PATH.open('a', encoding='utf-8') as error_file:
    for completed, (index, row) in enumerate(pending_rows, start=1):
        try:
            audio_array, audio_rate = extract_audio(row)
            asr_result = asr_pipe(
                {'raw': audio_array, 'sampling_rate': audio_rate},
                generate_kwargs={'language': 'Vietnamese', 'task': 'transcribe'},
            )
            transcript = asr_result['text'].strip()
            entities = extract_entities(transcript)
            reference = row.get('text', '') or ''
            sample_result = {
                'index': index,
                'utterance_id': row.get('utterance_id', ''),
                'audio_name': row.get('audio_name', ''),
                'reference_transcript': reference,
                'transcript': transcript,
                'wer': round(float(wer(reference, transcript)), 6) if reference and transcript else None,
                'duration': row.get('duration'),
                'entities': entities,
            }
            result_file.write(json.dumps(sample_result, ensure_ascii=False) + '\n')
            result_file.flush()
            results_by_index[index] = sample_result
            print('[{}/{}] index={} entities={} WER={}'.format(
                completed, len(pending_rows), index, len(entities), sample_result['wer']))
        except Exception as exc:
            error = {'index': index, 'error': repr(exc)}
            error_file.write(json.dumps(error, ensure_ascii=False) + '\n')
            error_file.flush()
            print('Bỏ qua index={}: {}'.format(index, exc))

results = [results_by_index[index] for index, _ in rows if index in results_by_index]
print('Hoàn tất phiên này sau {:.1f} giây; tổng kết quả: {}'.format(
    time.time() - started, len(results)))

Đã có kết quả checkpoint: 0


[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[1/500] index=0 entities=1 WER=0.1
[2/500] index=1 entities=3 WER=0.0
[3/500] index=2 entities=2 WER=0.0
[4/500] index=3 entities=2 WER=0.0
[5/500] index=4 entities=0 WER=0.0
[6/500] index=5 entities=2 WER=0.0
[7/500] index=6 entities=2 WER=0.0
[8/500] index=7 entities=6 WER=0.0
[9/500] index=8 entities=3 WER=0.0


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[10/500] index=9 entities=5 WER=0.310345
[11/500] index=10 entities=1 WER=0.0
[12/500] index=11 entities=2 WER=0.0
[13/500] index=12 entities=3 WER=0.0
[14/500] index=13 entities=0 WER=0.0
[15/500] index=14 entities=1 WER=0.0
[16/500] index=15 entities=3 WER=0.0
[17/500] index=16 entities=3 WER=0.0
[18/500] index=17 entities=4 WER=0.0
[19/500] index=18 entities=1 WER=0.0
[20/500] index=19 entities=7 WER=0.0
[21/500] index=20 entities=1 WER=0.0
[22/500] index=21 entities=1 WER=0.0
[23/500] index=22 entities=2 WER=0.0
[24/500] index=23 entities=1 WER=0.047619
[25/500] index=24 entities=2 WER=0.0
[26/500] index=25 entities=1 WER=0.185185
[27/500] index=26 entities=4 WER=0.0
[28/500] index=27 entities=3 WER=0.0
[29/500] index=28 entities=3 WER=0.222222
[30/500] index=29 entities=4 WER=0.0
[31/500] index=30 entities=3 WER=0.096774
[32/500] index=31 entities=4 WER=0.0
[33/500] index=32 entities=0 WER=0.0
[34/500] index=33 entities=2 WER=0.0
[35/500] index=34 entities=1 WER=0.0
[36/500] index

In [7]:
# Tổng hợp train results thành JSONL, CSV và summary
results_by_index = {}
if CHECKPOINT_PATH.exists():
    with CHECKPOINT_PATH.open(encoding='utf-8') as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                results_by_index[item['index']] = item
results = [results_by_index[index] for index in sorted(results_by_index)]

jsonl_path = OUTPUT_DIR / 'results.jsonl'
jsonl_path.write_text(
    ''.join(json.dumps(result, ensure_ascii=False) + '\n' for result in results),
    encoding='utf-8',
)
csv_path = OUTPUT_DIR / 'summary.csv'
with csv_path.open('w', encoding='utf-8-sig', newline='') as f:
    fields = ['index', 'utterance_id', 'audio_name', 'duration', 'wer', 'transcript', 'entity_count', 'entities']
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for result in results:
        writer.writerow({
            'index': result['index'],
            'utterance_id': result['utterance_id'],
            'audio_name': result['audio_name'],
            'duration': result['duration'],
            'wer': result['wer'],
            'transcript': result['transcript'],
            'entity_count': len(result['entities']),
            'entities': json.dumps(result['entities'], ensure_ascii=False),
        })

all_labels = Counter(entity['label'] for result in results for entity in result['entities'])
valid_wers = [result['wer'] for result in results if result['wer'] is not None]
summary = {
    'num_results': len(results),
    'num_errors': sum(1 for line in ERROR_PATH.read_text(encoding='utf-8').splitlines() if line.strip()) if ERROR_PATH.exists() else 0,
    'mean_wer': round(sum(valid_wers) / len(valid_wers), 6) if valid_wers else None,
    'median_wer': round(sorted(valid_wers)[len(valid_wers) // 2], 6) if valid_wers else None,
    'zero_wer_count': sum(value == 0 for value in valid_wers),
    'total_entities': sum(len(result['entities']) for result in results),
    'entity_label_counts': dict(all_labels),
    'asr_model': f'{ASR_REPO}/{ASR_SUBFOLDER}',
    'ner_model': f'{NER_REPO}/{NER_SUBFOLDER}',
}
summary_path = OUTPUT_DIR / 'summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Đã lưu:', jsonl_path, csv_path, summary_path)

{
  "num_results": 500,
  "num_errors": 0,
  "mean_wer": 0.041218,
  "median_wer": 0.0,
  "zero_wer_count": 406,
  "total_entities": 1278,
  "entity_label_counts": {
    "DISEASESYMTOM": 262,
    "MEDDEVICETECHNIQUE": 29,
    "TREATMENT": 95,
    "ORGAN": 309,
    "OCCUPATION": 65,
    "DATETIME": 81,
    "AGE": 122,
    "LOCATION": 26,
    "FOODDRINK": 3,
    "PERSONALCARE": 32,
    "SURGERY": 6,
    "PREVENTIVEMED": 29,
    "UNITCALIBRATOR": 77,
    "DIAGNOSTICS": 37,
    "DRUGCHEMICAL": 67,
    "GENDER": 38
  },
  "asr_model": "leduckhai/MultiMed-ST/asr/whisper-small-vietnamese/checkpoint-5000",
  "ner_model": "leduckhai/VietMed-NER/xlm-roberta-base-VietMed-NER"
}
Đã lưu: /content/multimed_results_500_train_rerun/results.jsonl /content/multimed_results_500_train_rerun/summary.csv /content/multimed_results_500_train_rerun/summary.json
